# PRÁCTICA COMPLETA — Examen 26 Mayo 2026

Versión expandida de PRÁCTICA.md con **Pipelines, GridSearchCV, cross-validation** y código paso a paso.

| Sección | Tipo |
|---|---|
| 1 | Workflow estándar con Pipeline |
| 2 | Regresión |
| 3 | Clasificación Binaria |
| 4 | Clasificación Multiclase |
| 5 | Clustering |
| 6 | Datos Faltantes |
| 7 | Normalización y Leakage |
| 8 | Overfitting, CV, GridSearch |
| 9 | Deep Learning (PyTorch MLP) |
| 10 | CNN |
| 11 | Transfer Learning |
| 12 | Hugging Face |


---
# 1. WORKFLOW ESTÁNDAR CON PIPELINE

El mayor error en examen es escalar antes del split o escalar test con parámetros de test.

`sklearn.pipeline.Pipeline` lo soluciona automáticamente: el `.fit()` del pipeline solo ve `X_train`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score
)

print('Imports OK')

In [ ]:
# TEMPLATE GENÉRICO — ajusta TARGET y el modelo final
# =====================================================

# --- PARÁMETROS A CAMBIAR ---
TARGET = 'target'
CSV_PATH = 'data.csv'
# ----------------------------

# 1. Cargar
df = pd.read_csv(CSV_PATH)

# 2. Inspección rápida
print('Shape:', df.shape)
print(df.head(3))
print('\nFaltantes:')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nTarget:')
print(df[TARGET].value_counts(dropna=False))

In [ ]:
# 3. Split PRIMERO (antes de cualquier transformación)
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # solo si es clasificación; quitar si es regresión
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# 4. Detectar columnas automáticamente
numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
categorical_cols = X_train.select_dtypes(exclude='number').columns.tolist()

print('Numéricas:', numeric_cols)
print('Categóricas:', categorical_cols)

In [ ]:
# 5. Construir Pipeline completo
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # NaN → mediana
    ('scaler', StandardScaler()),                    # z-score
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # NaN → moda
    ('encoder', OneHotEncoder(handle_unknown='ignore')),   # texto → dummies
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

# 6. Entrenar y evaluar
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

---
# 2. REGRESIÓN

Target es continuo (precio, temperatura, salario…)

Métricas clave:
- **R²**: proporción de varianza explicada (1 = perfecto, <0 = peor que la media)
- **RMSE**: en las mismas unidades que y, penaliza errores grandes
- **MAE**: en las mismas unidades que y, más robusto a outliers

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import GridSearchCV

# Datos reales (sin CSV)
housing = fetch_california_housing(as_frame=True)
df_reg = housing.frame
print(df_reg.shape)
print(df_reg.describe().round(2))

In [ ]:
X_reg = df_reg.drop(columns=['MedHouseVal'])
y_reg = df_reg['MedHouseVal']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
    # sin stratify porque target es continuo
)
print('Train:', X_train_r.shape)

In [ ]:
# Pipeline de regresión
reg_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])

reg_pipeline.fit(X_train_r, y_train_r)
y_pred_r = reg_pipeline.predict(X_test_r)

In [ ]:
# Métricas
mse  = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_r, y_pred_r)
r2   = r2_score(y_test_r, y_pred_r)

print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}  ← en mismas unidades que y')
print(f'MAE:  {mae:.4f}  ← robusto a outliers')
print(f'R²:   {r2:.4f}  ← >0.7 bueno, <0.5 malo')

In [ ]:
# Gráfica real vs predicho (obligatoria en examen de regresión)
plt.figure(figsize=(7, 5))
plt.scatter(y_test_r, y_pred_r, alpha=0.3, s=10)
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()], 'r--', lw=2)
plt.xlabel('Valor real')
plt.ylabel('Predicción')
plt.title(f'Real vs Predicho  (R²={r2:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# GridSearchCV para elegir alpha en Ridge/Lasso
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

grid_search = GridSearchCV(
    reg_pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_search.fit(X_train_r, y_train_r)

print('Mejor alpha:', grid_search.best_params_)
print('Mejor R² (CV):', grid_search.best_score_.round(4))

best_model = grid_search.best_estimator_
print('R² en test:', r2_score(y_test_r, best_model.predict(X_test_r)).round(4))

---
# 3. CLASIFICACIÓN BINARIA

Target tiene exactamente 2 clases (0/1, sí/no, spam/ham…)

| Métrica | Cuándo usarla |
|---|---|
| Accuracy | clases balanceadas |
| Precision | cuando FP es caro (spam, fraude: no acusar inocentes) |
| Recall | cuando FN es caro (enfermedad: no perder enfermos) |
| F1 | cuando no sabes cuál → equilibrio Precision/Recall |
| ROC-AUC | comparar modelos, robusta a desbalance |

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score, RocCurveDisplay

# Datos: cáncer de mama (benigno=1, maligno=0)
bc = load_breast_cancer(as_frame=True)
X_bc = bc.data
y_bc = bc.target

print('Distribución de clases:')
print(y_bc.value_counts())
print('(1=benigno, 0=maligno)')

In [ ]:
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

# Pipeline con LogisticRegression
clf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

clf_pipeline.fit(X_train_bc, y_train_bc)

y_pred_bc   = clf_pipeline.predict(X_test_bc)
y_proba_bc  = clf_pipeline.predict_proba(X_test_bc)[:, 1]  # prob clase positiva

In [ ]:
# Matriz de confusión con etiquetas explícitas
cm = confusion_matrix(y_test_bc, y_pred_bc)
#              pred=0   pred=1
# real=0  →  [ TN      FP ]
# real=1  →  [ FN      TP ]

tn, fp, fn, tp = cm.ravel()
print(f'Matriz de confusión:\n{cm}')
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print()
print(f'Accuracy:  {accuracy_score(y_test_bc, y_pred_bc):.4f}')
print(f'Precision: {precision_score(y_test_bc, y_pred_bc):.4f}  (TP/(TP+FP))')
print(f'Recall:    {recall_score(y_test_bc, y_pred_bc):.4f}  (TP/(TP+FN))')
print(f'F1:        {f1_score(y_test_bc, y_pred_bc):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test_bc, y_proba_bc):.4f}')

In [ ]:
# Curva ROC (útil para elegir threshold)
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test_bc, y_proba_bc, ax=ax)
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('Curva ROC')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation sobre todo X (más fiable que un solo split)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf_pipeline, X_bc, y_bc, cv=cv, scoring='f1')

print(f'F1 por fold: {scores.round(4)}')
print(f'Media: {scores.mean():.4f} ± {scores.std():.4f}')

---
# 4. CLASIFICACIÓN MULTICLASE

Más de 2 clases. Iris, MNIST, tipos de flores, etc.

En el `classification_report` cada fila es una clase. Presta atención al `macro avg` (trata todas igual) y `weighted avg` (pondera por soporte).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns

iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris = iris.target
class_names = iris.target_names

print('Clases:', class_names)
print('Distribución:', y_iris.value_counts().to_dict())

In [ ]:
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

rf_pipeline.fit(X_train_i, y_train_i)
y_pred_i = rf_pipeline.predict(X_test_i)

print(classification_report(y_test_i, y_pred_i, target_names=class_names))

In [ ]:
# Matriz de confusión como heatmap
cm_i = confusion_matrix(y_test_i, y_pred_i)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_i, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de confusión multiclase')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features del Random Forest
rf_model = rf_pipeline.named_steps['model']
importances = rf_model.feature_importances_
feature_names = X_iris.columns

importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=True)

importance_df.plot.barh(x='feature', y='importance', figsize=(6, 3), legend=False)
plt.title('Importancia de features')
plt.tight_layout()
plt.show()

---
# 5. CLUSTERING (No supervisado)

No hay etiquetas. El objetivo es descubrir grupos naturales.

**Pasos obligatorios:**
1. Elbow method → candidatos para K
2. Silhouette score → K óptimo (más cercano a 1)
3. Ajustar modelo final con K óptimo
4. Visualizar (PCA si hay más de 2 dimensiones)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs

# Datos sintéticos con 4 clusters reales
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
print('Forma:', X_blobs.shape)

In [ ]:
# Normalizar ANTES de clustering (KMeans usa distancias)
scaler_clust = StandardScaler()
X_blobs_scaled = scaler_clust.fit_transform(X_blobs)

# Elbow + Silhouette
K_range = range(2, 9)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_blobs_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_blobs_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method (busca el codo)')

axes[1].plot(list(K_range), sil_scores, 's-', color='orange')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette (más alto = mejor)')

plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f'Mejor K por silhouette: {best_k}')

In [ ]:
# Modelo final
km_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
labels_final = km_final.fit_predict(X_blobs_scaled)
centers_final = km_final.cluster_centers_

print(f'Silhouette final: {silhouette_score(X_blobs_scaled, labels_final):.4f}')
print(f'Inertia: {km_final.inertia_:.4f}')
print('Elementos por cluster:', np.bincount(labels_final))

In [ ]:
# Visualización (los datos ya son 2D, si fueran más usar PCA)
plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1],
                      c=labels_final, cmap='tab10', alpha=0.7, s=30)
plt.scatter(centers_final[:, 0], centers_final[:, 1],
            c='black', marker='X', s=200, label='Centroides')
plt.legend()
plt.title(f'KMeans K={best_k}')
plt.tight_layout()
plt.show()

In [ ]:
# PCA para visualizar datos de más dimensiones (ejemplo con iris)
pca = PCA(n_components=2)
X_iris_scaled = StandardScaler().fit_transform(X_iris)
X_iris_pca = pca.fit_transform(X_iris_scaled)

km_iris = KMeans(n_clusters=3, n_init=10, random_state=42)
labels_iris = km_iris.fit_predict(X_iris_scaled)

plt.figure(figsize=(6, 5))
plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=labels_iris, cmap='tab10', alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Iris clustering en espacio PCA')
plt.tight_layout()
plt.show()

print(f'Varianza explicada: {sum(pca.explained_variance_ratio_):.1%}')

---
# 6. DATOS FALTANTES

Tipos de missing:
- **MCAR** (Missing Completely At Random): borrar filas es seguro
- **MAR** (Missing At Random): imputa con media/mediana/KNN
- **MNAR** (Missing Not At Random): difícil, riesgo de sesgo

**Regla de oro:** siempre imputa dentro del Pipeline para que el fit sea solo en train.

In [ ]:
# Dataset con NaN artificial
np.random.seed(42)
n = 200
df_missing = pd.DataFrame({
    'edad':    np.random.randint(20, 70, n).astype(float),
    'ingresos': np.random.normal(30000, 10000, n),
    'ciudad':  np.random.choice(['Madrid', 'BCN', 'Valencia', None], n),
    'target':  np.random.randint(0, 2, n)
})

# Introduce NaN en edad e ingresos (~20%)
mask_edad     = np.random.rand(n) < 0.2
mask_ingresos = np.random.rand(n) < 0.15
df_missing.loc[mask_edad, 'edad'] = np.nan
df_missing.loc[mask_ingresos, 'ingresos'] = np.nan

print('Faltantes:')
print(df_missing.isnull().sum())
print(f'\nPorcentaje: {df_missing.isnull().mean().mul(100).round(1)}')

In [ ]:
# Estrategias de imputación en Pipeline
X_m = df_missing.drop(columns=['target'])
y_m = df_missing['target']

num_cols_m = ['edad', 'ingresos']
cat_cols_m = ['ciudad']

# Comparar: media vs mediana vs KNN
for strategy in ['mean', 'median']:
    pipe = Pipeline([
        ('pre', ColumnTransformer([
            ('num', Pipeline([('imp', SimpleImputer(strategy=strategy)), ('sc', StandardScaler())]), num_cols_m),
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore'))]), cat_cols_m),
        ])),
        ('clf', LogisticRegression(max_iter=500))
    ])
    cv_score = cross_val_score(pipe, X_m, y_m, cv=5, scoring='accuracy').mean()
    print(f'Imputación {strategy:8s} → CV Accuracy: {cv_score:.4f}')

# KNN requiere solo numéricas o todo numérico; aquí lo aplicamos por separado
from sklearn.impute import KNNImputer
knn_pipe = Pipeline([
    ('pre', ColumnTransformer([
        ('num', Pipeline([('imp', KNNImputer(n_neighbors=5)), ('sc', StandardScaler())]), num_cols_m),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore'))]), cat_cols_m),
    ])),
    ('clf', LogisticRegression(max_iter=500))
])
cv_knn = cross_val_score(knn_pipe, X_m, y_m, cv=5, scoring='accuracy').mean()
print(f'Imputación KNN      → CV Accuracy: {cv_knn:.4f}')

---
# 7. NORMALIZACIÓN Y DATA LEAKAGE

El leakage es el error más penalizado en examen.

**Regla:** `fit` solo en train → `transform` en train y test por separado.

El Pipeline lo hace automáticamente: cuando llamas `pipeline.fit(X_train)`, el scaler interno solo ve X_train.

In [ ]:
# Demostración del problema con leakage
from sklearn.datasets import make_classification

X_lk, y_lk = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train_lk, X_test_lk, y_train_lk, y_test_lk = train_test_split(
    X_lk, y_lk, test_size=0.2, random_state=42
)

# INCORRECTO: scaler ve todo X antes del split
scaler_bad = StandardScaler()
X_all_scaled = scaler_bad.fit_transform(X_lk)    # ← leakage: test contamina train
X_train_bad  = X_all_scaled[:800]
X_test_bad   = X_all_scaled[800:]

# CORRECTO: scaler solo ve X_train
scaler_ok = StandardScaler()
X_train_ok = scaler_ok.fit_transform(X_train_lk)
X_test_ok  = scaler_ok.transform(X_test_lk)      # ← usa parámetros de train

# La diferencia numérica puede parecer pequeña pero sesga la evaluación
print('Media test (con leakage):', X_test_bad.mean().round(6))
print('Media test (correcto):   ', X_test_ok.mean().round(6))
print('(Idénticamente 0 en correcto; no exactamente 0 con leakage)')

In [ ]:
# StandardScaler vs MinMaxScaler con y sin outliers
X_demo = np.array([[1], [2], [3], [4], [5], [100]])  # outlier en último

ss  = StandardScaler().fit_transform(X_demo)
mms = MinMaxScaler().fit_transform(X_demo)

compare = pd.DataFrame({
    'original': X_demo.ravel(),
    'StandardScaler': ss.ravel().round(3),
    'MinMaxScaler [0,1]': mms.ravel().round(3)
})
print(compare.to_string(index=False))
print('\nMinMaxScaler colapsa todo al inicio por el outlier 100')
print('StandardScaler es más robusto porque usa z-score')

In [ ]:
# RobustScaler → el más robusto a outliers (usa IQR)
from sklearn.preprocessing import RobustScaler

rs = RobustScaler().fit_transform(X_demo)

compare2 = pd.DataFrame({
    'original': X_demo.ravel(),
    'StandardScaler': ss.ravel().round(3),
    'RobustScaler (IQR)': rs.ravel().round(3)
})
print(compare2.to_string(index=False))

---
# 8. OVERFITTING, CROSS-VALIDATION Y GRIDSEARCHCV

**Señales de overfitting:**
- Train accuracy >> Test accuracy
- Curva de aprendizaje: train sube, val no sigue

**Remedios:**
- Reducir complejidad (max_depth, C, alpha)
- Regularización (Ridge/Lasso en regresión, C en LogisticRegression)
- Más datos o data augmentation
- Usar `Pipeline + GridSearchCV` para selección de hiperparámetros limpia

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import learning_curve, GridSearchCV

X_ov, y_ov = make_classification(n_samples=200, n_features=10, random_state=42)

# Comparación visual: árbol poco profundo vs muy profundo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, depth, label in zip(axes, [3, 20], ['max_depth=3 (simple)', 'max_depth=20 (overfitting)']):
    train_sizes, train_sc, val_sc = learning_curve(
        DecisionTreeClassifier(max_depth=depth, random_state=42),
        X_ov, y_ov, cv=5, train_sizes=np.linspace(0.1, 1.0, 10)
    )
    ax.plot(train_sizes, train_sc.mean(axis=1), label='Train')
    ax.plot(train_sizes, val_sc.mean(axis=1),   label='Validación')
    ax.fill_between(train_sizes,
                    train_sc.mean(axis=1) - train_sc.std(axis=1),
                    train_sc.mean(axis=1) + train_sc.std(axis=1), alpha=0.15)
    ax.fill_between(train_sizes,
                    val_sc.mean(axis=1) - val_sc.std(axis=1),
                    val_sc.mean(axis=1) + val_sc.std(axis=1), alpha=0.15)
    ax.set_title(label)
    ax.set_xlabel('Tamaño training')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.set_ylim(0.4, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# GridSearchCV dentro de un Pipeline → nunca hay leakage
X_train_ov, X_test_ov, y_train_ov, y_test_ov = train_test_split(
    X_ov, y_ov, test_size=0.2, random_state=42
)

pipe_gs = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

# Los parámetros del paso se acceden con 'nombre_paso__parámetro'
param_grid_rf = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth':    [3, 5, 10, None],
    'model__min_samples_leaf': [1, 5, 10]
}

gs = GridSearchCV(
    pipe_gs,
    param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
gs.fit(X_train_ov, y_train_ov)

print('Mejores parámetros:', gs.best_params_)
print('Mejor F1 (CV):', gs.best_score_.round(4))

In [ ]:
# Evaluación del mejor modelo en test
best_rf = gs.best_estimator_
y_pred_gs = best_rf.predict(X_test_ov)

print(classification_report(y_test_ov, y_pred_gs))
print('F1 en test:', f1_score(y_test_ov, y_pred_gs).round(4))

In [ ]:
# Resultados del grid como DataFrame para analizar
results_df = pd.DataFrame(gs.cv_results_)
top10 = results_df.sort_values('mean_test_score', ascending=False)[
    ['param_model__n_estimators', 'param_model__max_depth',
     'param_model__min_samples_leaf', 'mean_test_score', 'std_test_score']
].head(10)
print(top10.to_string(index=False))

---
# 9. DEEP LEARNING — PyTorch MLP

Pasos clave del loop de entrenamiento:
1. `optimizer.zero_grad()` — limpia gradientes del paso anterior
2. `outputs = model(data)` — forward pass
3. `loss = criterion(outputs, target)` — calcula error
4. `loss.backward()` — backpropagation
5. `optimizer.step()` — actualiza pesos

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Datos tabulares con PyTorch (sin torchvision)
bc = load_breast_cancer()
X_t = bc.data.astype(np.float32)
y_t = bc.target.astype(np.float32)

# Normalizar ANTES de crear tensors
scaler_t = StandardScaler()
X_t_scaled = scaler_t.fit_transform(X_t)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(X_t_scaled, y_t, test_size=0.2, random_state=42)

# Convertir a tensors
X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1).to(device)
X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)
y_te_t = torch.tensor(y_te, dtype=torch.float32).unsqueeze(1).to(device)

# DataLoader
train_ds = TensorDataset(X_tr_t, y_tr_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print('Train:', X_tr_t.shape, '| Test:', X_te_t.shape)

In [ ]:
# Definir MLP
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),       # Dropout para reducir overfitting
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()           # Salida binaria [0, 1]
        )

    def forward(self, x):
        return self.net(x)

model_mlp = MLP(input_dim=X_tr.shape[1]).to(device)
print(model_mlp)
print('Parámetros totales:', sum(p.numel() for p in model_mlp.parameters()))

In [ ]:
# Loss y Optimizer
criterion_mlp = nn.BCELoss()                               # Binary Cross Entropy
optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.001)

# Training loop con historial de loss
EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model_mlp.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        # 1. Limpiar gradientes
        optimizer_mlp.zero_grad()
        # 2. Forward pass
        outputs = model_mlp(X_batch)
        # 3. Loss
        loss = criterion_mlp(outputs, y_batch)
        # 4. Backward
        loss.backward()
        # 5. Actualizar pesos
        optimizer_mlp.step()

        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_loader))

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {train_losses[-1]:.4f}')

In [ ]:
# Curva de pérdida
plt.figure(figsize=(6, 3))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Curva de entrenamiento')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación en test
model_mlp.eval()
with torch.no_grad():
    proba_mlp = model_mlp(X_te_t).cpu().numpy().ravel()
    pred_mlp  = (proba_mlp >= 0.5).astype(int)

y_te_np = y_te_t.cpu().numpy().ravel().astype(int)

print(f'Accuracy: {accuracy_score(y_te_np, pred_mlp):.4f}')
print(f'F1:       {f1_score(y_te_np, pred_mlp):.4f}')
print(classification_report(y_te_np, pred_mlp))

---
# 10. CNN

Para imágenes. Arquitectura:
- **Conv2d** → detecta patrones locales (bordes, texturas)
- **MaxPool2d** → reduce dimensión, mantiene características dominantes
- **Flatten** → convierte feature maps a vector
- **Linear** → clasificación final

Cálculo de dimensiones: `output_size = (input_size + 2*padding - kernel_size) / stride + 1`

In [ ]:
from torchvision import datasets, transforms

# MNIST: 28x28 píxeles, 1 canal, 10 clases
transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # media y std de MNIST
])

train_dataset_m = datasets.MNIST('data', train=True,  download=True, transform=transform_mnist)
test_dataset_m  = datasets.MNIST('data', train=False, download=True, transform=transform_mnist)

train_loader_m = DataLoader(train_dataset_m, batch_size=64, shuffle=True)
test_loader_m  = DataLoader(test_dataset_m,  batch_size=64, shuffle=False)

print(f'Train batches: {len(train_loader_m)} | Test batches: {len(test_loader_m)}')

# Visualizar batch de ejemplo
examples = next(iter(train_loader_m))
images, labels = examples
print('Batch shape:', images.shape)  # [64, 1, 28, 28]

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, lbl in zip(axes.ravel(), images[:10], labels[:10]):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(lbl.item()))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Definir CNN
class CNN_MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        # Bloque convolucional 1: 1 canal → 32 filtros, 28x28 → 14x14
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        # Bloque convolucional 2: 32 → 64 filtros, 14x14 → 7x7
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Capa densa: 64 * 7 * 7 = 3136 → 128 → 10
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # [B, 32, 14, 14]
        x = self.pool(torch.relu(self.conv2(x)))   # [B, 64,  7,  7]
        x = x.view(x.size(0), -1)                  # Flatten → [B, 3136]
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)                            # logits sin softmax
        return x

cnn_model = CNN_MNIST().to(device)
print(cnn_model)
print('Parámetros:', sum(p.numel() for p in cnn_model.parameters() if p.requires_grad))

In [ ]:
# Training CNN (2 epochs para demo rápida)
criterion_cnn = nn.CrossEntropyLoss()  # incluye softmax internamente
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=0.001)

EPOCHS_CNN = 2
cnn_train_losses = []

for epoch in range(EPOCHS_CNN):
    cnn_model.train()
    running_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader_m):
        data, target = data.to(device), target.to(device)

        optimizer_cnn.zero_grad()
        outputs  = cnn_model(data)
        loss     = criterion_cnn(outputs, target)
        loss.backward()
        optimizer_cnn.step()

        running_loss += loss.item()

        if batch_idx % 200 == 0:
            print(f'Epoch {epoch+1} | Batch {batch_idx:4d} | Loss: {loss.item():.4f}')

    cnn_train_losses.append(running_loss / len(train_loader_m))

In [ ]:
# Evaluación CNN
cnn_model.eval()
correct = total = 0

with torch.no_grad():
    for data, target in test_loader_m:
        data, target = data.to(device), target.to(device)
        outputs    = cnn_model(data)
        _, predicted = torch.max(outputs, 1)     # argmax de logits
        total   += target.size(0)
        correct += (predicted == target).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

---
# 11. TRANSFER LEARNING

Tres estrategias:
1. **Feature extraction**: congela todo, entrena solo la última capa
2. **Fine-tuning parcial**: descongela las últimas capas
3. **Fine-tuning completo**: descongela todo (requiere muchos datos)

Para examen siempre usar estrategia 1 salvo que pidan otra.

In [ ]:
from torchvision import models

# Cargar ResNet18 preentrenada en ImageNet
resnet = models.resnet18(weights='IMAGENET1K_V1')  # 'pretrained=True' está deprecated

# Paso 1: Congelar TODOS los parámetros
for param in resnet.parameters():
    param.requires_grad = False

# Paso 2: Reemplazar la capa final (fc) por una nueva entreneable
num_features = resnet.fc.in_features        # 512 en ResNet18
NUM_CLASSES  = 10                           # tu número de clases
resnet.fc    = nn.Linear(num_features, NUM_CLASSES)

# Verificar qué se entrena
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet.parameters())
print(f'Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

resnet = resnet.to(device)

In [ ]:
# Solo el optimizador apunta a la capa entreneable
optimizer_tl = optim.Adam(resnet.fc.parameters(), lr=0.001)
criterion_tl = nn.CrossEntropyLoss()

# El training loop es idéntico al de la CNN
# resnet.train() → forward → loss → backward → optimizer.step()
print('Pipeline de Transfer Learning listo')
print('Transform necesario para ResNet18: 224x224, 3 canales, normalizar con ImageNet stats')
print('')
transform_imagenet = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
print('transform_imagenet definido')

In [ ]:
# Fine-tuning parcial: descongelar última capa convolucional
resnet2 = models.resnet18(weights='IMAGENET1K_V1')
for param in resnet2.parameters():
    param.requires_grad = False

# Descongelar layer4 (último bloque residual) + fc
for param in resnet2.layer4.parameters():
    param.requires_grad = True

resnet2.fc = nn.Linear(512, NUM_CLASSES)

trainable2 = sum(p.numel() for p in resnet2.parameters() if p.requires_grad)
total2     = sum(p.numel() for p in resnet2.parameters())
print(f'Con fine-tuning parcial: {trainable2:,} / {total2:,} parámetros entrenables ({100*trainable2/total2:.1f}%)')

# Usar lr diferente para capas preentrenadas y nuevas
optimizer_ft = optim.Adam([
    {'params': resnet2.layer4.parameters(), 'lr': 1e-4},  # lr pequeño para capas viejas
    {'params': resnet2.fc.parameters(),     'lr': 1e-3},  # lr mayor para capa nueva
])

---
# 12. HUGGING FACE

Para tareas de NLP con modelos preentrenados (BERT, DistilBERT, etc.)

- **pipeline()**: la forma más rápida, ideal para examen
- **AutoTokenizer + AutoModel**: más control, para fine-tuning

In [ ]:
# Nivel 1: Pipeline (forma más rápida)
from transformers import pipeline

# Sentiment analysis
sentiment = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

ejemplos = [
    'This movie is absolutely fantastic!',
    'I hated every minute of it.',
    'It was okay, not great not terrible.',
]

for texto, result in zip(ejemplos, sentiment(ejemplos)):
    print(f'{result["label"]:9s} ({result["score"]:.3f}) | {texto}')

In [ ]:
# Otros pipelines útiles
# Zero-shot classification (no necesita fine-tuning)
zs = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

result_zs = zs(
    'The stock market crashed today due to inflation fears.',
    candidate_labels=['sports', 'economy', 'politics', 'technology']
)

for label, score in zip(result_zs['labels'], result_zs['scores']):
    print(f'{label:12s}: {score:.3f}')

In [ ]:
# Nivel 2: AutoTokenizer + AutoModel (más control)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model_hf   = AutoModelForSequenceClassification.from_pretrained(model_name)

text   = 'Deep learning is revolutionizing AI.'
inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)

# Inferencia
with torch.no_grad():
    outputs = model_hf(**inputs)

logits   = outputs.logits
probs    = torch.softmax(logits, dim=-1)
pred_idx = logits.argmax().item()
labels   = ['NEGATIVE', 'POSITIVE']

print(f'Texto: {text}')
print(f'Predicción: {labels[pred_idx]} (prob: {probs[0, pred_idx]:.4f})')

In [ ]:
# RAG (Retrieval Augmented Generation) — pseudocódigo funcional
# En un exam puede pedirte describir los pasos, no necesariamente ejecutarlos

# Concepto: RAG = Retrieve + Augment + Generate
#
# 1. ÍNDICE: embeddings de documentos en una base vectorial
#    vectorstore = Chroma.from_documents(docs, embedding_model)
#
# 2. RETRIEVE: buscar los k documentos más similares a la query
#    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
#    relevant_docs = retriever.get_relevant_documents(query)
#
# 3. AUGMENT: construir el prompt con contexto
#    context = '\n'.join([d.page_content for d in relevant_docs])
#    prompt = f'Contexto:\n{context}\n\nPregunta: {query}\nRespuesta:'
#
# 4. GENERATE: el LLM genera la respuesta con contexto
#    response = llm(prompt)

print('Flujo RAG:')
print('Documento → Chunking → Embedding → VectorStore')
print('Query     → Embedding → Similarity Search → Top-K docs')
print('Top-K docs + Query → Prompt → LLM → Respuesta')

---
# CHECKLIST RÁPIDO DE EXAMEN

```
REGRESIÓN
  ☐ split → Pipeline(imputer, scaler, Ridge/Lasso)
  ☐ MSE, RMSE, MAE, R²
  ☐ Gráfica real vs predicho
  ☐ GridSearchCV para alpha

CLASIFICACIÓN BINARIA
  ☐ stratify en split
  ☐ Pipeline con class_weight='balanced'
  ☐ confusion_matrix + TP/TN/FP/FN
  ☐ Elegir métrica correcta (precision/recall/F1/AUC)
  ☐ cross_val_score para validar

MULTICLASE
  ☐ classification_report con target_names
  ☐ Heatmap de confusión
  ☐ Feature importance si es RandomForest

CLUSTERING
  ☐ Normalizar antes
  ☐ Elbow + Silhouette para elegir K
  ☐ PCA para visualizar >2D

DEEP LEARNING
  ☐ device = cuda if available
  ☐ DataLoader con batch_size y shuffle=True en train
  ☐ Loop: zero_grad → forward → loss → backward → step
  ☐ model.eval() + torch.no_grad() en evaluación

TRANSFER LEARNING
  ☐ weights='IMAGENET1K_V1' (no pretrained=True)
  ☐ Congelar todos → reemplazar fc
  ☐ Optimizer solo sobre fc.parameters()
  ☐ Transform 224x224 + normalización ImageNet
```